# MODELING SPECIES IN A BATCH REACTOR

### PROBLEM STATEMENT

We are interested in modeling cells ($X$), substrate ($S$), and product ($P$) concentration over time in a batch reactor as well as how the rates for growth ($r_g$), death ($r_d$), and maintenance ($r_{sm}$) change over time.

### IMPORTS

Import the packages necessary to run the ODEs and plot the outputs.

In [ ]:
import numpy as np
from math import exp
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from IPython.display import display, Math, Latex

### INITIAL VALUES

Input the range of volume the ODEs will be evaluated across.

In [ ]:
tmin = 0 
tmax = 12 # hr
tspan = np.array([tmin, tmax])
t_eval = np.linspace(tmin,tmax,num=200)

Input the initial values for the dependent variables (molar flow rates) and consolidate them into one vector.

In [ ]:
CX0 = 1.0 # g/dm^3
CS0 = 250.0 # g/dm^3
CP0 = 0.0 # g/dm^3
initial_values = [CX0, CS0, CP0]

### ODEs FUNCTION

Define the system of ODEs within a Python function. This will allow us to use solve_ivp() to solve this system of ODEs. The Python function cannot be split across multiple code cells.

In [ ]:
def ODEs(t, y):
    """
    The goal of this function is to set up the ODEs describing the change in
    each species in the batch reactor over time. Species of interest are cells (X),
    substrate (S), and product (P).
    """

    # Extract the concentrations from the y vector passed into the function
    CX = y[0] # g/dm^3
    CS = y[1] # g/dm^3
    CP = y[2] # g/dm^3

    # Default Parameters
    CPstar = 93 # g/dm^3
    n = 0.52
    umax = 0.46 # 1/hr
    KS = 33.5 # g/dm^3
    m = 0.03 # g substrate/(g cells-hr)
    kd = 0.01 # 1/hr
    YXS = 0.08 # g/g
    YSX = 1/YXS
    YPS = 0.45 # g/g
    YPX = 5.6 # g/g

    # Rate Laws (this is just to make plugging stuff in later easier)
    rg = (max(0, 1 - CP/CPstar))**n * umax*CS*CX/(KS + CS)
    rsm = m*CX
    rd = kd*CX
    rp = YPX*rg

    # Mass Balances (making use of the rate laws)
    dCXdt = rg - rd
    dCSdt = -YSX*rg - rsm - rp/YPS
    dCPdt = rp

    # Consolidate function output into a single vector and close the function
    return dCXdt, dCSdt, dCPdt

### SOLVE ODEs

Solve the system of ODEs contained in the ODEs function using solve_ivp().

In [ ]:
solution_output = solve_ivp(ODEs, tspan, initial_values, t_eval=t_eval, method='DOP853')

`solve_ivp` is the newer preferred solver for ODE problems. See [this webpage](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html#scipy.integrate.solve_ivp) for more details about the function and various methods and options that can be specified. Start with the default method (don't have that argument present or set it to 'RK45', and if your output doesn't look smooth, change the method until it does).

### EXTRACT SOLUTIONS

Extract individual solutions from output.

In [ ]:
t = solution_output.t
CX_soln = solution_output.y[0,:]
CS_soln = solution_output.y[1,:]
CP_soln = solution_output.y[2,:]

### PLOT CONCENTRATIONS

Now we want to plot our solution.

Note that we have different line styles and colors in the plots. The general plot formatting options are 
* Colors: black 'k', green 'g', red 'r', blue 'b', magenta 'm',  cyan blue 'c', yellow 'y'.
* Line types: '-' solid line, '--' dashed line, '-.' dash-dot line, ':' dotted line.
* Colors and line styles can be combined as 'r:' or 'k-.' for a red dotted line or a black dash-dot line.
* See this webpage for more details: https://matplotlib.org/2.1.2/api/_as_gen/matplotlib.pyplot.plot.html

In [ ]:
plt.figure(figsize=(6, 6))
line1, = plt.plot(t, CX_soln,'b-')
line2, = plt.plot(t, CS_soln, 'g--')
line3, = plt.plot(t, CP_soln, 'm-.')
plt.title('Profile of Concentrations of $C_{X}$, $C_{S}$, and $C_{P}$')
plt.legend((line1, line2, line3), ('$C_X$', '$C_S$', '$C_P$'))
plt.xlim(0,tmax)
plt.ylim(0, CS0)
plt.xlabel('$t\ (hr)$')
plt.ylabel('$C_{i} \ (g/dm^3)$')
plt.show()

### PLOT RATES

Copy and paste the all of the `# Default Parameters` and `# Raw Laws` code from your function (so the parameters have the same values and the rates have the same calculations--however note we are neglecting `rp`). But change every reference of `CX` to `CX_soln`, `CS` to `CS_soln`, and `CP` to `CP_soln`.

This will allow us to create plots of the rates over all time points, and it will use the value of each species concentraiton at that time point.

In [ ]:
# Default Parameters
CPstar = 93 # g/dm^3
n = 0.52
umax = 0.46 # 1/hr
KS = 33.5 # g/dm^3
m = 0.03 # g substrate/(g cells-hr)
kd = 0.01 # 1/hr
YXS = 0.08 # g/g
YSX = 1/YXS
YPS = 0.45 # g/g
YPX = 5.6 # g/g

# Rate Laws
rg = (np.maximum(0, 1 - CP_soln/CPstar))**n * umax*CS_soln*CX_soln/(KS + CS_soln)
rsm = m*CX_soln
rd = kd*CX_soln

In [ ]:
plt.figure(figsize=(6, 6))
line1, = plt.plot(t, rg,'b-')
line2, = plt.plot(t, rsm,'g--')
line3, = plt.plot(t, rd, 'm-.')
plt.title('Profile of Rates of $r_{g}$, $r_{sm}$, and $r_{d}$')
plt.legend((line1, line2, line3), ('$r_g$', '$r_{sm}$', '$r_d$'))
plt.xlim(0,tmax)
plt.ylim(0, 3)
plt.xlabel('$t\ (hr)$')
plt.ylabel('$r_{i} \ (g/dm^3 hr)$')
plt.show()